In [84]:
##importing packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.stats import ttest_ind
import statsmodels.formula.api as smf




In [85]:
#importing our dataset

hw_2_data = pd.read_csv("SCCS2_v12.csv")

print(hw_2_data.head())
print(hw_2_data.columns.tolist())
print(hw_2_data["gender"].value_counts())  
print(hw_2_data.isna().sum())

   Unnamed: 0  caseID  ethnic      height     weight      waist    hip    tc  \
0           1      23       1  177.350006  58.000000  69.500000  92.00  4.65   
1           2      39       1  173.350006  64.099998  75.550003  95.50  6.26   
2           3      42       1  177.100006  52.200001  63.000000  89.00  4.81   
3           4      51       1  167.800003  59.700001  72.000000  92.75  5.06   
4           5      68       1  172.300003  65.699997  72.500000  99.00  4.85   

     tg   hdl  ...  diabetes  hypertension  educ  drink  smoke  gender  \
0  0.54  1.07  ...         0             1   3.0      1      1       0   
1  1.20  1.20  ...         0             1   2.0      1      1       0   
2  0.79  1.64  ...         0             1   5.0      0      1       0   
3  2.19  1.13  ...         0             1   5.0      1      1       0   
4  1.35  1.15  ...         0             1   3.0      1      1       0   

   alcohol        age  ihd  Dummy2  
0        2  35.972622    0       2  


In [86]:
#problem 1 wants us to adjust for linnear and quadratic age, so lets create that quadratic age column first
#remember we also have to create our bmi colummn like last time

hw_2_data["age_squared"] = hw_2_data["age"] ** 2


height_meters = hw_2_data["height"] / 100
hw_2_data["bmi"] = hw_2_data["weight"] / (height_meters ** 2)

analytic = hw_2_data.dropna(subset=["tc", "gender", "bmi", "age"])



In [87]:
#ok now that we got our colmns made and our nice suset created, lets look at the models

# Model 1
y_total_cholesteral = analytic["tc"]

X1 = sm.add_constant(analytic[["gender", "age", "age_squared"]])
m1 = sm.OLS(y_total_cholesteral, X1).fit()

#now for our second model, lets add in BMI to see if it is a confounder

# Model 2
X2 = sm.add_constant(analytic[["gender", "age", "age_squared", "bmi"]])
m2 = sm.OLS(y_total_cholesteral, X2).fit()

print(m2.summary())

                            OLS Regression Results                            
Dep. Variable:                     tc   R-squared:                       0.166
Model:                            OLS   Adj. R-squared:                  0.160
Method:                 Least Squares   F-statistic:                     25.99
Date:                Thu, 24 Sep 2026   Prob (F-statistic):           1.15e-19
Time:                        21:09:20   Log-Likelihood:                -724.97
No. Observations:                 527   AIC:                             1460.
Df Residuals:                     522   BIC:                             1481.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           2.6224      0.414      6.335      

In [88]:
#Ok now we are making our third model. This time comparing bmi coefficnet before vs after gender is added
X3 = sm.add_constant(analytic[["bmi", "age", "age_squared"]])
m3 = sm.OLS(y_total_cholesteral, X3).fit()


# adding model 4 even though it should have the exact same variables as model 2
X4 = sm.add_constant(analytic[["bmi", "age", "age_squared", "gender"]])
m4 = sm.OLS(y_total_cholesteral, X4).fit()
print(m4.summary())

                            OLS Regression Results                            
Dep. Variable:                     tc   R-squared:                       0.166
Model:                            OLS   Adj. R-squared:                  0.160
Method:                 Least Squares   F-statistic:                     25.99
Date:                Thu, 24 Sep 2026   Prob (F-statistic):           1.15e-19
Time:                        21:09:20   Log-Likelihood:                -724.97
No. Observations:                 527   AIC:                             1460.
Df Residuals:                     522   BIC:                             1481.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           2.6224      0.414      6.335      

In [89]:
#ok now we are moving onto effect modification
#first we have to multiply gender and BMI into our gender_bmi column to actullly produce effect modification
analytic["gender_bmi"] = analytic["gender"] * analytic["bmi"]

#now we can add it to our model
X5 = sm.add_constant(analytic[["gender", "bmi", "gender_bmi", "age", "age_squared"]])
m5 = sm.OLS(y_total_cholesteral, X5).fit()
print(m5.summary())

                            OLS Regression Results                            
Dep. Variable:                     tc   R-squared:                       0.167
Model:                            OLS   Adj. R-squared:                  0.159
Method:                 Least Squares   F-statistic:                     20.85
Date:                Thu, 24 Sep 2026   Prob (F-statistic):           5.15e-19
Time:                        21:09:20   Log-Likelihood:                -724.76
No. Observations:                 527   AIC:                             1462.
Df Residuals:                     521   BIC:                             1487.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           2.4450      0.498      4.906      